# Veritas Protocol - Dilution Risk Score (DRS)

This notebook documents and reproduces the math behind Veritas: how a content asset's
authenticity becomes an on-chain risk score that a Uniswap v4 hook turns into a dynamic
LP fee and a pool gate. It mirrors the production code exactly:

- contract: `contracts/src/VeritasRegistry.sol` (`getCurrentDRS`, `getEffectiveD`, `saturateD`)
- oracle:   `oracle/src/drs.ts` (`combineDrs`, `duplicateDensity`, `dynamicFeeBps`)

**Live on Unichain Sepolia (1301):** Registry `0xe359bdB2cA1A152Ae62E2496F140ea192Ce0d824`,
Hook `0x0AaAef1B312243EfaAD238fb53403dC5761d60C4`. RSC on Reactive Lasna
`0xB44F024468dc78572D1Ad7b3f5Ce3A51408E5C5d`.

Run top-to-bottom. Only `numpy` and `matplotlib` are needed (no model downloads).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def clamp01(x):
    return np.clip(x, 0.0, 1.0)

## 1. The two risk channels, combined with noisy-OR

DRS blends two independent risks into one probability that a position is diluted:

- **A** = AI replicability (how easily the content is regenerated by a model)
- **D** = duplicate density (how many near-copies already exist)

$$\mathrm{DRS} = 1 - (1 - D)(1 - A)$$

This is a noisy-OR: either channel alone can drive risk high, and they reinforce.
On-chain the same formula runs in fixed-point on a 0..10000 scale.

In [ ]:
def combine_drs(d, a):
    return 1 - (1 - clamp01(d)) * (1 - clamp01(a))

for d, a in [(0.0, 0.0), (0.6, 0.0), (0.0, 0.95), (0.6, 0.95)]:
    print(f"D={d:.2f} A={a:.2f}  ->  DRS={combine_drs(d, a):.3f}")

## 2. The on-chain trustless D floor: `saturateD(dilutionCount)`

The duplicate channel has a fully on-chain, trustless component: a perceptual-hash
(pHash) near-duplicate count maintained autonomously by the Reactive Smart Contract.
It is mapped to a saturating risk with diminishing marginal effect (the 10th copy
matters less than the 2nd). This is the exact piecewise map from `VeritasRegistry.saturateD`.

In [ ]:
def saturate_d(count):
    if count == 0: return 0.0
    if count == 1: return 0.40
    if count == 2: return 0.60
    if count <= 3: return 0.75
    if count <= 5: return 0.85
    if count <= 10: return 0.90
    if count <= 20: return 0.95
    return 0.98

counts = np.arange(0, 25)
vals = [saturate_d(c) for c in counts]
plt.figure(figsize=(7, 3.5))
plt.step(counts, vals, where='post')
plt.axhline(0.85, ls='--', lw=1, label='pool gate (0.85)')
plt.xlabel('on-chain near-duplicate count'); plt.ylabel('D (on-chain floor)')
plt.title('saturateD: diminishing marginal risk per duplicate')
plt.legend(); plt.tight_layout(); plt.show()

## 3. The off-chain robust D: duplicate density from embeddings

pHash is blind to crops, rotations and AI re-style. The oracle covers that with a
neural signal: DINOv2 (and an independent CLIP) embeddings compared to a corpus.
Similarities above a per-embedder threshold accumulate into a density:

$$D = 1 - e^{-\lambda \sum_i s_i^{\beta}},\quad s_i \ge \tau$$

Production defaults: $\lambda = 0.7$, $\beta = 2.0$, $\tau = 0.55$ (DINOv2 space).

In [ ]:
LAMBDA_D, BETA, TAU = 0.7, 2.0, 0.55

def duplicate_density(sims, lam=LAMBDA_D, beta=BETA, tau=TAU):
    acc = sum(clamp01(s) ** beta for s in sims if s >= tau)
    return 1 - np.exp(-lam * acc)

examples = {
    'no matches':            [0.30, 0.41, 0.22],
    'one strong near-dup':   [0.92],
    'several near-dups':     [0.88, 0.91, 0.79, 0.85],
    'a crowd of copies':     [0.95]*8,
}
for name, sims in examples.items():
    print(f"{name:24s} -> D_offchain = {duplicate_density(sims):.3f}")

## 4. Effective D = max(on-chain floor, off-chain signal)

The two D sources are combined with `max()`, not an average. This hardens both failure modes:

- a **bribed oracle** under-reporting `offchainD` cannot push D below the trustless on-chain floor;
- the **blind pHash** missing a cropped/rotated copy is covered by the off-chain neural signal.

Neither side can hide risk; the on-chain channel is the trust anchor, the off-chain channel the sensitivity.

In [ ]:
def effective_d(dilution_count, offchain_sims):
    return max(saturate_d(dilution_count), duplicate_density(offchain_sims))

def drs(dilution_count, offchain_sims, a):
    return combine_drs(effective_d(dilution_count, offchain_sims), a)

# A bribed oracle reports no off-chain matches, but 2 on-chain pHash copies exist:
print('honest off-chain only:', round(drs(0, [0.95, 0.90], a=0.0), 3))
print('bribed (no off-chain), but on-chain floor holds:', round(drs(2, [], a=0.0), 3))

## 5. DRS to dynamic LP fee, and the gate

The hook prices impermanent-loss risk linearly in DRS between a base and max fee, and
rejects pool creation entirely above the gate:

$$\text{fee} = \text{base} + (\text{max} - \text{base})\cdot \mathrm{DRS},\quad \text{gate at } 0.85$$

base = 0.30%, max = 1.00%, gate = 0.85.

In [ ]:
BASE_BPS, MAX_BPS, GATE = 3000, 10000, 0.85  # hundredths of a bip

def dynamic_fee_pct(drs_val):
    bps = BASE_BPS + (MAX_BPS - BASE_BPS) * clamp01(drs_val)
    return bps / 1e4  # to percent

xs = np.linspace(0, 1, 200)
fees = [dynamic_fee_pct(x) for x in xs]
plt.figure(figsize=(7, 3.5))
plt.plot(xs, fees)
plt.axvspan(GATE, 1.0, color='red', alpha=0.12, label='gated (no pool)')
plt.xlabel('DRS'); plt.ylabel('LP fee (%)')
plt.title('Dynamic LP fee vs DRS'); plt.legend(); plt.tight_layout(); plt.show()

## 6. LP Insurance Reserve: the second DRS-driven layer

The hook has **two** DRS-driven mechanisms, not one. The first (above) is the dynamic LP fee
(`beforeSwap`). The second is a per-pool insurance buffer (`afterSwap`):

- **Reserve skim:** each swap donates a DRS-scaled slice of the output to a per-pool buffer
  held as ERC-6909 claims by the hook. Formula:

$$\text{reserveFee} = \text{MAX\_RESERVE\_FEE} \times \mathrm{DRS}$$

where MAX_RESERVE_FEE = 2500 (0.25% ceiling, scaled by DRS).

- **Disbursement trigger:** the buffer is paid back to in-range LPs (via `PoolManager.donate`)
  only when DRS has risen at least DISBURSE_TRIGGER_DELTA above its value at pool creation:

$$\mathrm{DRS}_\text{current} \ge \mathrm{DRS}_\text{creation} + 0.15$$

This is the "verified dilution event": the hook pays out exactly when LPs are hurt.
A low-DRS pool accrues almost nothing and pays nothing out (no charge on safe content).
A high-DRS pool accrues more and can claim a real payout if dilution is confirmed.

Mirror of `VeritasHook.sol`: `MAX_RESERVE_FEE = 2500`, `DISBURSE_TRIGGER_DELTA = 1500`.

In [ ]:
MAX_RESERVE_FEE_BPS = 2500   # 0.25% ceiling
DISBURSE_TRIGGER_DELTA = 1500  # 0.15 in 0-10000 scale

def reserve_skim_pct(drs_val):
    bps = MAX_RESERVE_FEE_BPS * clamp01(drs_val)
    return bps / 1e4

def total_lp_cost_pct(drs_val):
    return dynamic_fee_pct(drs_val) + reserve_skim_pct(drs_val)

xs = np.linspace(0, 1, 200)
lp_fees   = [dynamic_fee_pct(x) for x in xs]
reserves  = [reserve_skim_pct(x) for x in xs]
totals    = [total_lp_cost_pct(x) for x in xs]

plt.figure(figsize=(7, 3.5))
plt.fill_between(xs, 0, lp_fees, alpha=0.4, label='LP fee (beforeSwap)')
plt.fill_between(xs, lp_fees, totals, alpha=0.4, label='reserve skim (afterSwap)')
plt.axvspan(GATE, 1.0, color='red', alpha=0.10, label='gated')
plt.xlabel('DRS'); plt.ylabel('cost per swap (%)')
plt.title('Two DRS-driven layers: LP fee + insurance reserve skim')
plt.legend(); plt.tight_layout(); plt.show()

# Disbursement example: pool created at DRS 0.00, dilution event raises it to 0.68
drs_creation, drs_current = 0.00, 0.68
trigger = (drs_current - drs_creation) >= DISBURSE_TRIGGER_DELTA / 1e4
print(f"drsAtCreation={drs_creation:.2f}  drs_current={drs_current:.2f}")
print(f"delta={drs_current - drs_creation:.2f} >= {DISBURSE_TRIGGER_DELTA/1e4:.2f}  ->  disburse eligible: {trigger}")

## 7. The three demo tiers (real on-chain attestations)

These are real attestations on Unichain Sepolia (`contracts/deployments/seeded-dataset.json`).

In [ ]:
tiers = [
    ('A unique photo  0xf87daf', 0.00, 0.00),
    ('B some dup+AI    0xfe3119', 0.49, 0.376),
    ('C AI-replicable  0x627552', 0.00, 0.95),
]
for name, d, a in tiers:
    s = combine_drs(d, a)
    status = 'GATED' if s > GATE else f'pool @ {dynamic_fee_pct(s):.2f}%'
    print(f"{name}  D={d:.2f} A={a:.2f}  DRS={s:.2f}  -> {status}")

## 8. The living-D loop (proven live, no keeper)

We attested the same image under two owners (a near-duplicate pair). Attesting the second
triggered the `DilutionMonitorRSC` on Reactive Lasna; its callback on Unichain ran
`getNearDuplicates` + `incrementDilutionCount`, raising the first attestation's on-chain
`dilutionCount` from 0 to 2 with no oracle and no off-chain keeper. The cell below shows
how that count alone moves effective D via the trustless floor.

In [ ]:
for c in [0, 1, 2]:
    print(f"dilutionCount={c}  ->  on-chain floor D = {saturate_d(c):.2f}")
print('\nWATCH attestation A: 0x7d07e4ea40a445aed2fafb3ebe7559f9a003f691b8e754a1a3a531b04cc5e0a6')
print('observed on-chain: dilutionCount 0 -> 2, effective D 0.00 -> 0.60 (DilutionIncremented events on Uniscan)')

## References

- Mekacher et al., *Scientific Reports* (Nature), 2022: rarer NFTs carry measurably lower downside (3.7M+ transactions).
- Production code: `oracle/src/drs.ts`, `contracts/src/VeritasRegistry.sol`.
- Live addresses + the living-D proof: repo `README.md` and `contracts/deployments/`.